# 04. Preprocessing Validation Summary

## 목적

이 노트북은 `pandas-preprocessing-cases`의 최종 QA 리포트다.

앞선 3개 노트북의 전처리 결과를 통합해, 각 데이터가 의도한 분석 단위와 품질 기준을 만족하는지 확인한다.

검증 대상:

- 01. 이커머스 랭킹 상품 데이터 정제
- 02. Olist 다중 테이블 base table 설계
- 03. 기업 리뷰 텍스트 전처리와 카테고리 신호 추출

이 노트북의 목적은 단순히 행 수와 결측치를 세는 것이 아니라, **전처리 결과가 실제 포트폴리오에서 사용 가능한 상태인지 PASS / CHECK / LIMITATION 기준으로 판정하는 것**이다.

In [ ]:
import pandas as pd
from pathlib import Path

INPUT_DIR = Path("../outputs")
OUTPUT_DIR = Path("../outputs/04_validation")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

d01 = INPUT_DIR / "01_product"
d02 = INPUT_DIR / "02_olist"
d03 = INPUT_DIR / "03_review"

# 01 outputs
raw_cleaned = pd.read_csv(d01 / "01_raw_cleaned_comparison.csv")
ranking_type = pd.read_csv(d01 / "01_ranking_type_summary.csv")
platform_summary = pd.read_csv(d01 / "01_platform_preprocessing_summary.csv")

# 02 outputs
naive_join = pd.read_csv(d02 / "02_naive_join_risk_summary.csv")
pre_agg = pd.read_csv(d02 / "02_pre_aggregation_summary.csv")
order_base_val = pd.read_csv(d02 / "02_order_base_validation.csv")
item_base_val = pd.read_csv(d02 / "02_item_base_validation.csv")
question_mapping = pd.read_csv(d02 / "02_analysis_question_table_mapping.csv")

# 03 outputs
company_summary = pd.read_csv(d03 / "03_company_category_signal_summary.csv")
coverage = pd.read_csv(d03 / "03_category_coverage_summary.csv")
unmatched_sample = pd.read_csv(d03 / "03_unmatched_review_sample.csv")

print("Loaded 01, 02, 03 preprocessing outputs.")

## 1. 검증 기준

이 노트북에서는 검증 상태를 세 가지로 구분한다.

- `PASS`: 포트폴리오에서 바로 설명 가능한 상태
- `CHECK`: 값 자체는 생성되었지만 해석 시 추가 설명이 필요한 상태
- `LIMITATION`: 한계로 명시해야 하는 상태

전처리 결과는 항상 100% 완벽한 데이터가 아니라, **무엇을 말할 수 있고 무엇을 말하면 안 되는지 구분할 수 있어야** 포트폴리오에서 안전하게 사용할 수 있다.

In [ ]:
validation_rows = []

def add(area, check_item, result, status, interpretation):
    validation_rows.append({
        "validation_area": area,
        "check_item": check_item,
        "result": result,
        "status": status,
        "interpretation": interpretation
    })

## 2. 01 상품 데이터 전처리 검증

검증 포인트:

- 원본 통합 데이터와 전처리 데이터의 행 수 비교
- 랭킹 화면 기준 데이터와 고유 상품 기준 데이터 분리 여부
- 랭킹 기준별 순위 범위와 결측 확인
- product_key 기반 중복 관리 여부

In [ ]:
def get_metric(df, key):
    row = df[df["check_item"].eq(key)] if "check_item" in df.columns else pd.DataFrame()
    if len(row) == 0:
        return None
    return row["value"].iloc[0]

raw_rows = get_metric(raw_cleaned, "raw_rows")
cleaned_rows = get_metric(raw_cleaned, "cleaned_rows")
ranking_view_rows = get_metric(raw_cleaned, "ranking_view_rows")
unique_product_rows = get_metric(raw_cleaned, "unique_product_rows")

add("product_data", "raw_to_cleaned_rows", f"raw={raw_rows}, cleaned={cleaned_rows}", "PASS",
    "원본 통합 데이터와 기본 전처리 데이터의 행 수를 비교해 전처리 단계의 기준 행 수를 확인했다.")

add("product_data", "ranking_view_and_unique_product_split", f"ranking_view={ranking_view_rows}, unique_product={unique_product_rows}", "PASS",
    "랭킹 화면 기준 분석과 고유 상품 기준 분석을 분리했다.")

rank_range_ok = bool(((ranking_type["rank_min"] >= 1) & (ranking_type["rank_max"] <= 100)).all())
add("product_data", "ranking_rank_range", f"rank_min/max within 1~100: {rank_range_ok}", "PASS" if rank_range_ok else "CHECK",
    "랭킹 기준별 순위 범위가 의도한 수집 범위와 맞는지 확인했다.")

missing_cols = [c for c in ["missing_price", "missing_review_count", "missing_rating"] if c in ranking_type.columns]
total_missing = int(ranking_type[missing_cols].sum().sum()) if missing_cols else 0
add("product_data", "ranking_missing_values", f"missing total={total_missing}", "PASS" if total_missing == 0 else "CHECK",
    "가격, 리뷰 수, 평점 결측 여부를 랭킹 기준별로 점검했다.")

dup_overall = int(platform_summary["duplicate_overall_count"].sum()) if "duplicate_overall_count" in platform_summary.columns else 0
add("product_data", "duplicate_product_management", f"duplicate_overall_count={dup_overall}", "PASS",
    "중복 상품을 삭제만 한 것이 아니라 ranking_view와 unique_product 기준으로 분리해 관리했다.")

product_validation = pd.DataFrame([r for r in validation_rows if r["validation_area"] == "product_data"])
product_validation.to_csv(OUTPUT_DIR / "04_product_validation_summary.csv", index=False, encoding="utf-8-sig")
product_validation

## 3. 02 Olist base table 검증

검증 포인트:

- 주문 단위 base table에서 `order_id`가 중복되지 않는지
- 상품 단위 base table에서 `order_id` 중복은 정상인지
- `order_id + order_item_id` 조합이 고유한지
- naive JOIN이 왜 위험한지
- 사전 집계 테이블이 존재하는지

In [ ]:
order_dup = int(order_base_val["duplicate_order_id_count"].iloc[0])
add("olist_base_table", "order_base_order_id_uniqueness", f"duplicate_order_id_count={order_dup}", "PASS" if order_dup == 0 else "CHECK",
    "주문 단위 base table에서 order_id가 중복되지 않는지 확인했다.")

item_dup_order = int(item_base_val["duplicate_order_id_count"].iloc[0])
item_dup_key = int(item_base_val["duplicate_order_item_key_count"].iloc[0])
add("olist_base_table", "item_base_order_item_grain", f"duplicate_order_id_count={item_dup_order}, duplicate_order_item_key_count={item_dup_key}", "PASS" if item_dup_key == 0 else "CHECK",
    "상품 단위 base table에서 order_id 중복은 정상이며, order_id+order_item_id 조합이 고유한지 확인했다.")

naive_rows = int(naive_join["naive_join_rows"].iloc[0])
distinct_orders = int(naive_join["distinct_orders_after_join"].iloc[0])
add("olist_base_table", "naive_join_row_increase", f"naive_join_rows={naive_rows}, distinct_orders={distinct_orders}", "PASS",
    "원본 테이블을 한 번에 JOIN하면 행 수가 증가해 주문 단위 KPI가 왜곡될 수 있음을 확인했다.")

add("olist_base_table", "pre_aggregation_tables", f"pre_aggregation_tables={len(pre_agg)}", "PASS",
    "결제와 리뷰를 주문 단위로 사전 집계한 테이블을 확인했다.")

add("olist_base_table", "analysis_question_mapping", f"mapping_rows={len(question_mapping)}", "PASS",
    "분석 질문별로 주문 단위/상품 단위 base table 선택 기준을 정리했다.")

olist_validation = pd.DataFrame([r for r in validation_rows if r["validation_area"] == "olist_base_table"])
olist_validation.to_csv(OUTPUT_DIR / "04_olist_validation_summary.csv", index=False, encoding="utf-8-sig")
olist_validation

## 4. 03 리뷰 텍스트 전처리 검증

검증 포인트:

- 원본 리뷰가 long format으로 변환되었는지
- 실제 사전 파일이 로드되었는지
- 카테고리 매칭률이 계산되었는지
- 미매칭 리뷰 샘플이 분리되었는지
- 기업 단위 카테고리 요약 테이블이 생성되었는지

In [ ]:
def coverage_metric(metric):
    row = coverage[coverage["metric"].eq(metric)]
    if len(row) == 0:
        return None
    return row["value"].iloc[0]

raw_review_rows = coverage_metric("raw_review_rows")
text_segment_rows = coverage_metric("text_segment_rows")
match_rate = coverage_metric("category_match_rate_percent")
unmatched_count = coverage_metric("unmatched_text_segment_count")
dict_terms = coverage_metric("category_dictionary_terms")

add("review_text", "long_format_conversion", f"raw_review_rows={raw_review_rows}, text_segment_rows={text_segment_rows}", "PASS",
    "리뷰제목, 장점, 단점, 경영진 의견을 long format으로 변환해 텍스트 유형별 분석이 가능해졌다.")

add("review_text", "dictionary_loaded", f"category_dictionary_terms={dict_terms}", "PASS",
    "실제 카테고리 사전을 불러와 리뷰 텍스트 매핑에 사용했다.")

add("review_text", "category_match_rate", f"category_match_rate_percent={match_rate}", "PASS" if float(match_rate) >= 50 else "CHECK",
    "전체 텍스트 중 카테고리 1개 이상 매칭된 비율을 확인했다. 이 값은 사전 기반 방식의 커버리지로 해석한다.")

add("review_text", "unmatched_review_sample", f"unmatched_text_segment_count={unmatched_count}, sample_rows={len(unmatched_sample)}", "LIMITATION" if float(unmatched_count) > 0 else "PASS",
    "미매칭 텍스트 샘플을 별도로 확인해 사전 기반 방식의 한계를 명시했다.")

add("review_text", "company_signal_summary", f"company_summary_rows={len(company_summary)}", "PASS",
    "리뷰 단위 카테고리 신호를 기업 단위 요약 테이블로 집계했다.")

review_validation = pd.DataFrame([r for r in validation_rows if r["validation_area"] == "review_text"])
review_validation.to_csv(OUTPUT_DIR / "04_review_validation_summary.csv", index=False, encoding="utf-8-sig")
review_validation

## 5. 통합 검증 요약

01·02·03 검증 결과를 하나의 테이블로 통합한다.

In [ ]:
integrated_validation = pd.DataFrame(validation_rows)
integrated_validation.to_csv(OUTPUT_DIR / "04_integrated_validation_summary.csv", index=False, encoding="utf-8-sig")
integrated_validation

## 6. 포트폴리오 사용 가능 여부 판정

각 구성 요소를 포트폴리오에 어떻게 사용할지 정리한다.

In [ ]:
readiness_check = pd.DataFrame([
    {
        "portfolio_component": "01_product_data_cleaning",
        "readiness": "READY",
        "main_evidence": "실제 이커머스 랭킹 상품 CSV 기반. raw→cleaned→ranking_view→unique_product 흐름과 랭킹 기준별 품질 점검 포함.",
        "caution": "product_key는 완전한 상품 식별자가 아니라 중복 완화 기준으로 설명해야 함."
    },
    {
        "portfolio_component": "02_olist_base_table_design",
        "readiness": "READY",
        "main_evidence": "실제 Olist DB 기반. 테이블 grain, naive JOIN 위험, 사전 집계, 주문/상품 base table 검증 포함.",
        "caution": "order_base와 item_base의 기준 단위 차이를 명확히 설명해야 함."
    },
    {
        "portfolio_component": "03_review_text_preprocessing",
        "readiness": "READY_WITH_LIMITATION",
        "main_evidence": "실제 잡플래닛 리뷰 원문과 실제 사전 3종 기반. long format, 표준화, 카테고리 매핑, 기업 단위 요약 포함.",
        "caution": "사전 기반 방식은 문맥 반전/비꼼/사전 미등록 표현을 완벽히 처리하지 못하므로 보조 지표로 설명해야 함."
    },
    {
        "portfolio_component": "04_preprocessing_validation_summary",
        "readiness": "READY",
        "main_evidence": "01~03 결과물을 통합해 PASS/CHECK/LIMITATION 기준으로 품질 검증.",
        "caution": "이 노트북은 분석 결과가 아니라 전처리 QA 리포트로 설명해야 함."
    },
])

readiness_check.to_csv(OUTPUT_DIR / "04_final_portfolio_readiness_check.csv", index=False, encoding="utf-8-sig")
readiness_check

## 최종 요약

이 통합 검증 노트북의 결론은 다음과 같다.

1. **상품 데이터 정제**는 실제 이커머스 랭킹 상품 데이터 기반으로 구성되었으며, 랭킹 화면 기준 데이터와 고유 상품 기준 데이터를 분리했다.
2. **Olist base table 설계**는 실제 SQLite DB 기반으로 검증되었으며, 주문 단위와 상품 단위 테이블을 분리해 중복 집계 위험을 줄였다.
3. **리뷰 텍스트 전처리**는 실제 잡플래닛 리뷰 원문과 실제 사전 파일을 사용했으며, 리뷰 텍스트를 기업 단위 카테고리 신호로 구조화했다.
4. 사전 기반 리뷰 매핑에는 미매칭 표현과 문맥 해석 한계가 있으므로, 결과는 자동 판정 지표가 아니라 추천/진단 판단을 보조하는 신호로 해석해야 한다.

따라서 `pandas-preprocessing-cases`는 단순 pandas 문법 예제가 아니라, 실제 프로젝트 데이터의 전처리 설계·실행·검증 과정을 보여주는 Skill Evidence로 사용할 수 있다.